In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor, Pool

from src.config import (
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\pc\Desktop\yzta-2026-datathon


In [2]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)

In [3]:
def add_features_v2(df):
    df = df.copy()

    eps = 1e-6

    # Temel uyku feature'ları
    if {"rem_yuzdesi", "derin_uyku_yuzdesi"}.issubset(df.columns):
        df["toplam_kaliteli_uyku_yuzdesi"] = df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
        df["rem_derin_uyku_carpim"] = df["rem_yuzdesi"] * df["derin_uyku_yuzdesi"]
        df["rem_derin_uyku_orani"] = df["rem_yuzdesi"] / (df["derin_uyku_yuzdesi"] + eps)

    # Uyku bozulması
    if {"gecelik_uyanma_sayisi", "uykuya_dalma_suresi_dk"}.issubset(df.columns):
        df["uyku_bolunme_yuku"] = df["gecelik_uyanma_sayisi"] * df["uykuya_dalma_suresi_dk"]
        df["uyku_verimsizlik_skoru"] = df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
        df["uyanma_kare"] = df["gecelik_uyanma_sayisi"] ** 2
        df["uykuya_dalma_kare"] = df["uykuya_dalma_suresi_dk"] ** 2

    # Stres / çalışma yükü
    if {"stres_skoru", "gunluk_calisma_saati"}.issubset(df.columns):
        df["stres_calisma_yuku"] = df["stres_skoru"] * df["gunluk_calisma_saati"]
        df["stres_kare"] = df["stres_skoru"] ** 2
        df["calisma_saati_kare"] = df["gunluk_calisma_saati"] ** 2
        df["stres_per_calisma"] = df["stres_skoru"] / (df["gunluk_calisma_saati"] + eps)

    # Ekran / kafein
    if {"uyku_oncesi_ekran_suresi_dk", "uyku_oncesi_kafein_mg"}.issubset(df.columns):
        df["ekran_kafein_yuku"] = df["uyku_oncesi_ekran_suresi_dk"] + df["uyku_oncesi_kafein_mg"]
        df["ekran_kafein_carpim"] = df["uyku_oncesi_ekran_suresi_dk"] * df["uyku_oncesi_kafein_mg"]

    if {"uyku_oncesi_ekran_suresi_dk", "uykuya_dalma_suresi_dk"}.issubset(df.columns):
        df["ekran_per_uykuya_dalma"] = (
            df["uyku_oncesi_ekran_suresi_dk"] / (df["uykuya_dalma_suresi_dk"] + eps)
        )

    if {"uyku_oncesi_kafein_mg", "stres_skoru"}.issubset(df.columns):
        df["kafein_per_stres"] = df["uyku_oncesi_kafein_mg"] / (df["stres_skoru"] + eps)

    # Aktivite / fizyoloji
    if "gunluk_adim_sayisi" in df.columns:
        df["adim_sayisi_bin"] = df["gunluk_adim_sayisi"] / 1000

    if {"dinlenik_nabiz_bpm", "stres_skoru"}.issubset(df.columns):
        df["nabiz_stres_yuku"] = df["dinlenik_nabiz_bpm"] * df["stres_skoru"]

    # BMI kategori
    if "vucut_kitle_indeksi" in df.columns:
        df["bmi_kategori"] = pd.cut(
            df["vucut_kitle_indeksi"],
            bins=[0, 18.5, 25, 30, np.inf],
            labels=["zayif", "normal", "kilolu", "obez"]
        ).astype("object")

    # Hafta sonu flag
    if "gun_tipi" in df.columns:
        df["hafta_sonu_flag"] = (df["gun_tipi"] == "Hafta sonu").astype(int)

    # Ruh sağlığı ordinal risk
    if "ruh_sagligi_durumu" in df.columns:
        risk_map = {
            "Saglikli": 0,
            "Anksiyete": 1,
            "Depresyon": 2,
            "Anksiyete ve depresyon": 3,
        }
        df["ruh_sagligi_risk_skoru"] = df["ruh_sagligi_durumu"].map(risk_map)

    # Kategorik interaction feature'lar
    if {"kronotip", "gun_tipi"}.issubset(df.columns):
        df["kronotip_gun_tipi"] = df["kronotip"].astype(str) + "_" + df["gun_tipi"].astype(str)

    if {"meslek", "gun_tipi"}.issubset(df.columns):
        df["meslek_gun_tipi"] = df["meslek"].astype(str) + "_" + df["gun_tipi"].astype(str)

    if {"ruh_sagligi_durumu", "gun_tipi"}.issubset(df.columns):
        df["ruh_gun_tipi"] = df["ruh_sagligi_durumu"].astype(str) + "_" + df["gun_tipi"].astype(str)

    if {"kronotip", "mevsim"}.issubset(df.columns):
        df["kronotip_mevsim"] = df["kronotip"].astype(str) + "_" + df["mevsim"].astype(str)

    return df

In [4]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())

Train shape: (56000, 24)
Test shape: (24000, 23)
Sample submission shape: (2, 2)


,id,yas,cinsiyet,meslek,vucut_kitle_indeksi,ulke,rem_yuzdesi,derin_uyku_yuzdesi,uykuya_dalma_suresi_dk,gecelik_uyanma_sayisi,...,stres_skoru,gunluk_calisma_saati,kronotip,ruh_sagligi_durumu,dinlenik_nabiz_bpm,oda_sicakligi_celsius,hafta_sonu_uyku_farki_saat,mevsim,gun_tipi,bilissel_performans_skoru
0,1,34,Erkek,Saglik Personeli,31.470103,Cin,14.431210,14.645436,27,7,...,9.922976,10.045274,Sabah insani,Anksiyete ve depresyon,78,18.962436,-0.074140,Sonbahar-Kis,Hafta ici,0.136441
1,2,32,Kadin,Muhendis,30.981394,Amerika,21.771870,27.220360,20,4,...,6.626400,6.319245,Gece insani,Saglikli,76,21.225666,0.942672,Sonbahar-Kis,Hafta ici,5.848312
2,3,39,Erkek,Ev Hanimi,21.533898,Spain,18.178857,25.530104,33,7,...,6.093566,3.824463,Notr,Depresyon,66,18.482409,1.239886,Ilkbahar-Yaz,Hafta sonu,6.828276
3,4,40,Kadin,Egitimci,23.310749,Yeni Zelanda,21.438151,15.891188,21,2,...,3.168185,4.597316,Gece insani,Saglikli,60,21.862235,0.727695,Sonbahar-Kis,Hafta sonu,8.144649
4,5,36,Kadin,NaN,NaN,Portekiz,25.468018,16.356738,21,8,...,7.198574,3.189120,Notr,Anksiyete ve depresyon,74,19.223195,-0.223402,Sonbahar-Kis,Hafta ici,0.431423


,id,yas,cinsiyet,meslek,vucut_kitle_indeksi,ulke,rem_yuzdesi,derin_uyku_yuzdesi,uykuya_dalma_suresi_dk,gecelik_uyanma_sayisi,...,sekerleme_suresi_dk,stres_skoru,gunluk_calisma_saati,kronotip,ruh_sagligi_durumu,dinlenik_nabiz_bpm,oda_sicakligi_celsius,hafta_sonu_uyku_farki_saat,mevsim,gun_tipi
0,1,42,Erkek,Saglik Personeli,29.728724,Ingiltere,21.691917,23.990157,21,5,...,0,6.349196,9.419561,Gece insani,Depresyon,82,21.627129,1.716161,Ilkbahar-Yaz,Hafta sonu
1,2,26,Kadin,Serbest Calisan,32.865996,Cin,22.090624,18.231963,34,5,...,0,NaN,8.574199,Gece insani,Saglikli,66,27.835934,1.482283,Ilkbahar-Yaz,Hafta ici
2,3,21,Erkek,Lojistik Calisani,24.438264,Cin,19.488438,16.695363,17,4,...,52,6.715192,10.519017,Notr,Depresyon,63,20.252258,1.981034,Sonbahar-Kis,Hafta ici
3,4,61,Kadin,Saglik Personeli,23.275057,Cin,26.831092,21.762040,29,2,...,2,5.633400,9.814789,Gece insani,Saglikli,71,17.680971,0.589750,Sonbahar-Kis,Hafta ici
4,5,26,Kadin,Saglik Personeli,24.815531,Ingiltere,16.271522,18.153212,15,4,...,0,7.953794,9.308398,Sabah insani,Saglikli,53,23.698979,0.622211,Ilkbahar-Yaz,Hafta ici


In [5]:
train_fe = add_features_v2(train)
test_fe = add_features_v2(test)

print("Train before:", train.shape)
print("Train after:", train_fe.shape)

print("Test before:", test.shape)
print("Test after:", test_fe.shape)

Train before: (56000, 24)
Train after: (56000, 48)
Test before: (24000, 23)
Test after: (24000, 47)


In [6]:
X = train_fe.drop(columns=[TARGET, ID_COL])
y = train_fe[TARGET]

X_test = test_fe.drop(columns=[ID_COL])
test_ids = test_fe[ID_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (56000, 46)
y shape: (56000,)
X_test shape: (24000, 46)


In [7]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", len(cat_cols))
print(cat_cols)

print("Numeric columns:", len(num_cols))

Categorical columns: 12
['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi', 'bmi_kategori', 'kronotip_gun_tipi', 'meslek_gun_tipi', 'ruh_gun_tipi', 'kronotip_mevsim']
Numeric columns: 34


In [8]:
X_cb = X.copy()
X_test_cb = X_test.copy()

# Kategorik eksikler
for col in cat_cols:
    X_cb[col] = X_cb[col].astype("object").fillna("Missing").astype(str)
    X_test_cb[col] = X_test_cb[col].astype("object").fillna("Missing").astype(str)

# Sayısal eksikler
for col in num_cols:
    median_value = X_cb[col].median()
    X_cb[col] = X_cb[col].fillna(median_value)
    X_test_cb[col] = X_test_cb[col].fillna(median_value)

cat_features = [X_cb.columns.get_loc(col) for col in cat_cols]

print("Cat feature indices:", cat_features)

Cat feature indices: [1, 2, 4, 15, 16, 20, 21, 39, 42, 43, 44, 45]


In [9]:
native_catboost_experiments = {
    "catboost_native_depth4": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=4,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_native_depth5": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_native_depth4_l2_5": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=4,
        l2_leaf_reg=5,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "catboost_native_depth4_lr002": CatBoostRegressor(
        iterations=4000,
        learning_rate=0.02,
        depth=4,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),
}

In [10]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

def run_native_catboost(model_name, model, X, y, X_test, cat_features):
    print("=" * 80)
    print(f"Model: {model_name}")

    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), N_SPLITS))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        print(f"Fold {fold}")

        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        train_pool = Pool(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features
        )

        valid_pool = Pool(
            X_valid_fold,
            y_valid_fold,
            cat_features=cat_features
        )

        test_pool = Pool(
            X_test,
            cat_features=cat_features
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=False
        )

        valid_pred = model.predict(valid_pool)
        valid_pred = np.clip(valid_pred, 0, 10)

        fold_rmse = rmse(y_valid_fold, valid_pred)
        fold_scores.append(fold_rmse)

        oof_pred[valid_idx] = valid_pred

        test_pred = model.predict(test_pool)
        test_pred = np.clip(test_pred, 0, 10)
        test_pred_folds[:, fold - 1] = test_pred

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    mean_rmse = np.mean(fold_scores)
    std_rmse = np.std(fold_scores)

    print(f"{model_name} CV RMSE: {mean_rmse:.5f} ± {std_rmse:.5f}")

    return {
        "model": model_name,
        "cv_rmse_mean": mean_rmse,
        "cv_rmse_std": std_rmse,
        "fold_scores": fold_scores,
        "oof_pred": oof_pred,
        "test_pred": test_pred_folds.mean(axis=1)
    }

In [11]:
native_results = []
native_oof_predictions = {}
native_test_predictions = {}

for model_name, model in native_catboost_experiments.items():
    result = run_native_catboost(
        model_name=model_name,
        model=model,
        X=X_cb,
        y=y,
        X_test=X_test_cb,
        cat_features=cat_features
    )

    native_results.append({
        "model": result["model"],
        "cv_rmse_mean": result["cv_rmse_mean"],
        "cv_rmse_std": result["cv_rmse_std"],
        "fold_scores": result["fold_scores"]
    })

    native_oof_predictions[model_name] = result["oof_pred"]
    native_test_predictions[model_name] = result["test_pred"]

Model: catboost_native_depth4
Fold 1
Fold 1 RMSE: 1.22063
Fold 2
Fold 2 RMSE: 1.21777
Fold 3
Fold 3 RMSE: 1.20266
Fold 4
Fold 4 RMSE: 1.20953
Fold 5
Fold 5 RMSE: 1.23218
catboost_native_depth4 CV RMSE: 1.21655 ± 0.01004
Model: catboost_native_depth5
Fold 1
Fold 1 RMSE: 1.22130
Fold 2
Fold 2 RMSE: 1.21994
Fold 3
Fold 3 RMSE: 1.20370
Fold 4
Fold 4 RMSE: 1.21069
Fold 5
Fold 5 RMSE: 1.23385
catboost_native_depth5 CV RMSE: 1.21790 ± 0.01024
Model: catboost_native_depth4_l2_5
Fold 1
Fold 1 RMSE: 1.22106
Fold 2
Fold 2 RMSE: 1.21770
Fold 3
Fold 3 RMSE: 1.20300
Fold 4
Fold 4 RMSE: 1.20990
Fold 5
Fold 5 RMSE: 1.23256
catboost_native_depth4_l2_5 CV RMSE: 1.21685 ± 0.01005
Model: catboost_native_depth4_lr002
Fold 1
Fold 1 RMSE: 1.22144
Fold 2
Fold 2 RMSE: 1.21790
Fold 3
Fold 3 RMSE: 1.20257
Fold 4
Fold 4 RMSE: 1.20964
Fold 5
Fold 5 RMSE: 1.23216
catboost_native_depth4_lr002 CV RMSE: 1.21674 ± 0.01012


In [12]:
native_results_df = pd.DataFrame(native_results)
native_results_df = native_results_df.sort_values("cv_rmse_mean").reset_index(drop=True)

native_results_df[["model", "cv_rmse_mean", "cv_rmse_std"]]

,model,cv_rmse_mean,cv_rmse_std
0,catboost_native_depth4,1.216555,0.010045
1,catboost_native_depth4_lr002,1.216740,0.010124
2,catboost_native_depth4_l2_5,1.216846,0.010054
3,catboost_native_depth5,1.217896,0.010237


In [13]:
best_native_model = native_results_df.loc[0, "model"]
best_native_pred = native_test_predictions[best_native_model]

best_native_pred = np.clip(best_native_pred, 0, 10)

submission_native = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: best_native_pred
})

print("Best native model:", best_native_model)
print("Submission shape:", submission_native.shape)
display(submission_native.head())

Best native model: catboost_native_depth4
Submission shape: (24000, 2)


,id,bilissel_performans_skoru
0,1,5.989866
1,2,6.434160
2,3,3.037792
3,4,7.112258
4,5,3.621443


In [14]:
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

native_submission_path = SUBMISSION_DIR / f"submission_{best_native_model}.csv"

submission_native.to_csv(native_submission_path, index=False)

print("Saved:", native_submission_path)

Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_catboost_native_depth4.csv


In [15]:
PREDICTION_DIR = PROJECT_ROOT / "predictions" / "catboost_native"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

for model_name in native_oof_predictions.keys():
    oof_df = pd.DataFrame({
        ID_COL: train[ID_COL],
        "y_true": y,
        "oof_pred": np.clip(native_oof_predictions[model_name], 0, 10)
    })

    test_df = pd.DataFrame({
        ID_COL: test_ids,
        "test_pred": np.clip(native_test_predictions[model_name], 0, 10)
    })

    oof_path = PREDICTION_DIR / f"{model_name}_oof.csv"
    test_path = PREDICTION_DIR / f"{model_name}_test.csv"

    oof_df.to_csv(oof_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Saved:", oof_path)
    print("Saved:", test_path)

Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth4_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth4_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth5_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth5_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth4_l2_5_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth4_l2_5_test.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth4_lr002_oof.csv
Saved: c:\Users\pc\Desktop\yzta-2026-datathon\predictions\catboost_native\catboost_native_depth4_lr002_test.csv


In [16]:
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

native_results_path = RESULTS_DIR / "catboost_native_results.csv"

native_results_df.to_csv(native_results_path, index=False)

print("Saved:", native_results_path)
display(native_results_df)

Saved: c:\Users\pc\Desktop\yzta-2026-datathon\results\catboost_native_results.csv


,model,cv_rmse_mean,cv_rmse_std,fold_scores
0,catboost_native_depth4,1.216555,0.010045,"[1.220630884153273, 1.2177691098105143, 1.2026..."
1,catboost_native_depth4_lr002,1.216740,0.010124,"[1.2214357947223018, 1.217895712994533, 1.2025..."
2,catboost_native_depth4_l2_5,1.216846,0.010054,"[1.221064592855108, 1.217700006566504, 1.20300..."
3,catboost_native_depth5,1.217896,0.010237,"[1.2213016636049452, 1.2199388839492429, 1.203..."
